# Co-piloto Jurídico Híbrido
## Recuperación y análisis de precedentes penales del Supremo Tribunal de Justicia del Estado de Jalisco
### Documentación técnica y metodológica · versión 11.2

**Contenido**

1. Planteamiento del problema, objeto y alcance
2. Arquitectura del sistema, modelos y equipo de cómputo
3. Fase I. Obtención del corpus
4. Fase II. Procesamiento y depuración
5. Composición del corpus penal
6. Fase III. Indexación híbrida y consumo de memoria
7. Fase IV. Flujo de análisis de una consulta
8. Catálogo de instrucciones al modelo (*prompts*)
9. Fase V. Evaluación
10. Confidencialidad, alcance y líneas de desarrollo
11. Referencias

**Glosario de términos técnicos** (se definen aquí para no interrumpir la exposición):

| Término | Significado |
|---|---|
| Modelo de lenguaje (LLM) | Programa entrenado con grandes volúmenes de texto que redacta y extrae información a partir de instrucciones |
| *Prompt* | Instrucción escrita que se entrega al modelo de lenguaje junto con el texto que debe analizar |
| RAG | *Retrieval-Augmented Generation*: el modelo redacta solo a partir de documentos recuperados previamente, no de su memoria |
| Token | Unidad mínima de texto que procesa un modelo; en español, de 3 a 4 caracteres en promedio |
| Ventana de contexto | Cantidad máxima de tokens que el modelo puede leer en una sola instrucción (aquí, 8,192) |
| Embedding | Representación del significado de un texto como una lista de números (vector); textos de sentido parecido quedan cercanos |
| Coseno | Medida de cercanía entre dos vectores: 1 indica el mismo sentido |
| Fragmento y traslape | Porción de una resolución que se indexa por separado; el traslape es el texto compartido entre fragmentos contiguos |
| ChromaDB / HNSW | Base de datos local de vectores / estructura que permite encontrar los vectores más cercanos sin compararlos todos |
| BM25 / IDF | Búsqueda por coincidencia de palabras / peso mayor para los términos poco frecuentes en el corpus |
| RRF | *Reciprocal Rank Fusion*: combina dos listas de resultados según la posición de cada documento en ellas |
| Expresión regular | Patrón que localiza texto con una forma determinada (p. ej. «Notifíquese») |
| Lema | Forma base de una palabra («robaron» → «robar»); se usa solo en el clasificador |
| Precisión / exhaustividad | De lo que el sistema afirma, fracción correcta / de lo que debía encontrar, fracción encontrada |
| JSON | Formato de datos estructurado (campo: valor) que el sistema exige al modelo para poder verificar la ficha |
| Cuantización | Reducción de la precisión numérica de un modelo para que ocupe menos memoria (aquí, 4 bits) |
| GPU / VRAM | Procesador gráfico que ejecuta los modelos / su memoria propia (8 GB en este equipo) |
| float16 | Número en media precisión: ocupa la mitad que la precisión simple (float32) |
| QC_1 / QC_2 | Controles de calidad: relevancia del candidato / coherencia del delito de la ficha |
| Familia de delito | Agrupación de tipos penales afines (p. ej. robo calificado y robo equiparado) |
| Juez LLM | Modelo de lenguaje distinto del de la aplicación que lee las resoluciones para construir la referencia de evaluación |
| Kappa de Cohen | Acuerdo entre dos lectores descontando el que se esperaría por azar (0 = azar, 1 = total) |
| IC 95 % | Intervalo de confianza: rango en que se encuentra el valor real con 95 % de confianza |
| Hit@k, MRR, nDCG | Métricas de búsqueda: si hay un resultado útil entre los k primeros, en qué posición aparece el primero y la calidad del orden |
| Partición de desarrollo / prueba | Consultas para ajustar el sistema / consultas reservadas para medirlo una sola vez |
| Ollama / Streamlit | Servidor local de modelos de lenguaje / biblioteca con que se construyó la interfaz web |

Este documento se ejecuta sobre el código del paquete `copiloto_juridico`, el mismo que utilizan la aplicación web
y la evaluación; no reproduce código propio. Las cifras de las celdas de resultados se calculan al ejecutarlo; las
que aparecen en el texto citan la corrida guardada en `evaluacion/resultados/` que las respalda.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from copiloto_juridico import config

os.chdir(config.ROOT_DIR)  # las rutas son relativas a la raíz del repositorio
pd.set_option("display.max_colwidth", 110)
llm = config.LLMSettings()
print(f"Modelo de lenguaje: {llm.model} · temperatura {llm.temperature} · semilla {llm.seed} · "
      f"ventana de contexto {llm.num_ctx:,} tokens")
print(f"Modelo de embeddings: {config.EMBEDDING_MODEL_NAME} · materias en la búsqueda: {', '.join(config.MATERIAS_ACTIVAS)}")

Modelo de lenguaje: llama3.1:8b · temperatura 0.0 · semilla 42 · ventana de contexto 8,192 tokens
Modelo de embeddings: intfloat/multilingual-e5-base · materias en la búsqueda: penal


## 1. Planteamiento del problema, objeto y alcance

**Planteamiento.** Quien prepara o contesta un recurso de apelación penal y el propio tribunal de alzada que lo
resuelve requieren conocer el criterio con que las Salas Penales del Supremo Tribunal de Justicia del Estado
de Jalisco (STJ) han resuelto supuestos análogos.
El portal público del tribunal permite consultar las resoluciones por categoría, pero no por similitud de hechos ni
por problema procesal, y cada resolución debe leerse íntegra para identificar el acto impugnado, los agravios, su
calificación y el sentido del fallo.

**Contexto.** En 2024, el 89.42 % de los casos que el sistema de justicia penal mexicano debía resolver no obtuvo una
solución efectiva; de 627 mil causas penales acumuladas en los tribunales, el 10.55 % alcanzó una resolución efectiva,
y entre 2019 y 2024 las causas acumuladas pasaron de 221 mil a 425 mil [12]. México ocupa el lugar 135 de 143 países
en el factor de justicia penal del *Rule of Law Index 2025* [13]. En Jalisco se registraron en 2024 14,677 personas
procesadas o imputadas en causas penales ingresadas en primera instancia, séptimo lugar nacional; de las personas
imputadas con resolución del juez de control, el 76.3 % fue vinculada a proceso [14]. En ese contexto de carga y
rezago, consultar el criterio de las Salas sobre supuestos análogos debe ser tan económico como sea posible para
quienes intervienen en el proceso.

**Objeto.** Desarrollar un sistema que, a partir de la descripción de un caso en lenguaje natural, recupere las
resoluciones del propio tribunal más afines y elabore de cada una una ficha estructurada (delito, acto impugnado,
apelante, agravios y su calificación, sentido del fallo, fundamento legal), así como una síntesis comparativa que
indique la parte favorecida por cada precedente.

**Usuarios destinatarios.** Dos perfiles ante el mismo problema de localización y lectura de precedentes, que
emplean la misma búsqueda y las mismas fichas:

| Perfil | Necesidad |
|---|---|
| Abogado, defensor o asesor jurídico de la víctima (usuario principal) | Interponer o contestar un recurso de apelación con precedentes en que la Sala calificó agravios semejantes |
| Tribunal de alzada (Sala Penal) | Contrastar su criterio con el de resoluciones previas en supuestos semejantes; el sentido de la resolución le corresponde exclusivamente |

**Alcance.** Materia penal exclusivamente. La consulta de una persona sin formación jurídica queda fuera del alcance
de esta versión: no tiene perfil en la aplicación ni se ha evaluado. El sistema no pronostica el resultado del asunto, no redacta agravios y
no sustituye la lectura íntegra de la resolución ni el criterio profesional. Las resoluciones del STJ constituyen
precedentes orientadores, no jurisprudencia obligatoria.

**Método.** Generación aumentada por recuperación (*Retrieval-Augmented Generation*, RAG): el modelo de lenguaje
redacta exclusivamente a partir del texto de las resoluciones recuperadas, no de su conocimiento general.

## 2. Arquitectura del sistema, modelos y equipo de cómputo

![Arquitectura del sistema](diagramas/arquitectura.svg)

Todos los componentes se ejecutan en el equipo local; ninguna consulta ni dato del caso se transmite a servicios
externos.

### 2.1 Modelos utilizados y razón de su elección

| Componente | Modelo | Tamaño | Función | Razón de la elección |
|---|---|---|---|---|
| Modelo de lenguaje de la aplicación | `llama3.1:8b` (Meta), cuantizado Q4_K_M, servido por Ollama | 4.9 GB en disco; 6.9 GB cargado con ventana de 8,192 tokens | conceptos clave, QC_1, ficha, síntesis | Cabe íntegro en la GPU de 8 GB con la ventana necesaria para leer una resolución cuando el modelo de embeddings se ejecuta en CPU (sección 6); uso local permitido por su licencia; salida JSON restringida |
| Modelo de embeddings | `intfloat/multilingual-e5-base` | ~278 M parámetros, 768 dimensiones, 512 tokens | representación semántica de fragmentos y consultas | Entrenado en ~100 idiomas, incluido el español (el modelo previo, `all-mpnet-base-v2`, era de inglés); de los mejor calificados de su tamaño en el banco de pruebas MTEB multilingüe; cabe en memoria junto al modelo de lenguaje o se ejecuta en CPU |
| Modelo de respaldo de las fichas | `qwen2.5:7b` (Alibaba) | 4.7 GB | fichas precalculadas de las resoluciones sin ficha del modelo principal | Cabe en 8 GB; 18 s por ficha frente a 29 s y JSON válido 20/20 en la comparación de modelos |
| Lector de contraste en la evaluación | `gemma2:9b` (Google) | 5.4 GB | lectura independiente de cada resolución y de la relevancia | Familia de modelos distinta de la aplicación; acreditó el umbral de exactitud |
| Juez evaluado | `mistral:7b` (Mistral AI) | 4.4 GB | lectura de contraste | Exactitud 0.795 en 1,412 resoluciones frente al umbral de 0.80 (corrida `20261117-065346_a60ca4f_referencias`) |
| Juez evaluado en piloto | `qwen2.5:7b` (Alibaba) | 4.7 GB | — | Prueba piloto de 60 resoluciones por debajo del umbral; se aprovecha como modelo complementario de fichas |
| Alternativa por recursos | `qwen2.5:14b` (Alibaba) | 9.0 GB | — | Requiere más de los 8 GB de memoria de la GPU |

**Criterios de selección del modelo de lenguaje de la aplicación.**

1. *Requisito de memoria (excluyente).* El modelo debe caber íntegro en los 8 GB de la GPU con la ventana de 8,192
   tokens necesaria para leer una resolución; si una parte se desborda a la RAM, cada llamada se vuelve mucho más
   lenta. Quedan excluidos `gemma2:9b` (10.2 GB cargado, de los cuales 2.2 GB en RAM, según Ollama) y `qwen2.5:14b`
   (9.0 GB solo en disco). Cumplen el requisito `llama3.1:8b` (6.9 GB cargado según `ollama ps`, 6.3 GiB en la
   GPU según `nvidia-smi`, con el modelo de embeddings en CPU) y `qwen2.5:7b`. Si el modelo de embeddings ocupa la GPU
   antes que el de lenguaje, Ollama coloca en ella las capas que caben y el resto en la RAM (medición de la sección 6).
2. *Entre los que cumplen: continuidad y costo del cambio.* `llama3.1:8b` es el modelo con el que se desarrollaron y
   ajustaron las instrucciones, la ficha JSON, las regresiones y las 2,495 fichas precalculadas.

**llama3.1:8b frente a qwen2.5:7b.** En la comparación de modelos locales
(corrida `20261113-125326_fb1f3fe_modelos`: mismas 10 consultas, mismas 20 fichas, semilla 42, sin referencia),
qwen2.5:7b resultó más rápido y mejor en los diagnósticos:

| Modelo | Latencia p50 por consulta | s por ficha (p50) | JSON válido | Sentido = reglas | Artículos no encontrados | «Dato no disponible» | VRAM máxima |
|---|---|---|---|---|---|---|---|
| llama3.1:8b | 125 s | 28.8 | 19/20 | 4/6 | 11.5 % | 8.9 % | 7,311 MiB |
| qwen2.5:7b | 83 s | 18.3 | 20/20 | 5/6 | 6.6 % | 3.5 % | 7,049 MiB |

**Análisis de impacto de sustituir el modelo de lenguaje.**

| Componente | ¿Depende del modelo de lenguaje? | Consecuencia de sustituirlo |
|---|---|---|
| Corpus, depuración, embeddings, ChromaDB, BM25 y RRF | No | Se conservan |
| Referencia de la evaluación (categoría del STJ, reglas, juez `gemma2:9b`) | No | Se conserva |
| Paso 1, conceptos clave | Sí | Cambian los conceptos y, con ellos, la clasificación y la búsqueda: repetir las regresiones; recalcular la caché de conceptos de la evaluación (su clave no incluye el modelo) |
| QC_1 | Sí | Repetir `qc` |
| Ficha (paso 4) | Sí | Las 2,495 fichas precalculadas dejan de ser vigentes (su huella incluye el modelo): regenerarlas (~12.7 h) y repetir `fichas-diagnostico` y `fichas-referencia`; el tercer intento de la ficha se diseñó sobre las omisiones de llama3.1:8b |
| Síntesis (paso 5) | Sí | Repetir la regresión y la verificación de citas |
| Latencia | Sí | Repetir `latencia` |

**Evidencia disponible.** qwen2.5:7b destaca en velocidad y en completitud de campos en el diagnóstico de 20 fichas;
en la prueba piloto como lector de la familia del delito quedó por debajo del umbral de 0.80. La configuración vigente
aprovecha ambas fortalezas.

**Decisión: uso complementario de ambos modelos.** El modelo principal de la aplicación y de
toda la evaluación es `llama3.1:8b`, con el que se construyeron y midieron las 2,495 fichas. `qwen2.5:7b` se incorpora como **modelo de respaldo de las fichas**
(`config.FICHAS_MODELO_RESPALDO`): `fichas-batch --respaldo` precalcula con él, por lotes y sin otro modelo cargado,
las fichas de las resoluciones que carecen de una ficha válida de `llama3.1:8b`, fuera de la muestra de la referencia
(la evaluación sigue midiendo solo el modelo principal). La aplicación utiliza primero la ficha del modelo principal
y, en su defecto, la de respaldo, en lugar de generarla durante la consulta; el modelo de cada ficha queda registrado.
El usuario no realiza ninguna acción adicional. La sustitución completa del modelo principal se decidirá con este
protocolo: (1) fichas de la muestra de la referencia
con `qwen2.5:7b` (725 resoluciones, unas 3.7 horas); (2) `fichas-referencia` y `qc` en la partición de desarrollo
con ambos modelos; (3) adopción si mejora la exactitud con un intervalo de confianza que excluya el empate, seguida de
la regeneración completa de las fichas y de la evaluación.

Los tamaños en disco son los que reporta `ollama list`; la ocupación en memoria se midió con `ollama ps` y
`nvidia-smi` (6.3 de 8 GiB de la GPU con `llama3.1:8b` cargado). En razón de esa ocupación, **nunca se cargan dos
modelos de lenguaje a la vez** y, mientras el modelo de lenguaje ocupa la GPU, el modelo de embeddings se ejecuta en
la CPU.

### 2.2 Equipo de cómputo

| Elemento | Especificación |
|---|---|
| Equipo | ASUS ROG Zephyrus G14 (2024), modelo GA403UV [2] |
| Procesador | AMD Ryzen 9 8945HS, 8 núcleos / 16 hilos |
| Memoria | 32 GB LPDDR5X |
| GPU | NVIDIA GeForce RTX 4060 Laptop, 8 GB GDDR6, arquitectura Ada Lovelace (AD107), 3,072 núcleos CUDA, 35–115 W [3] |
| Costo de referencia | USD 1,599 de precio de lista; ofertado en USD 1,299.99 [1] |
| Servidor de modelos | Ollama 0.6.5 |

**Razones para la ejecución local.** (i) Confidencialidad: la descripción del caso, amparada por el secreto
profesional, no sale del equipo; (ii) costo marginal nulo por consulta una vez adquirido el equipo;
(iii) reproducibilidad: temperatura 0, semilla fija y huella (*digest*) del modelo registrada en cada corrida.

## 3. Fase I. Obtención del corpus

![Fase I](diagramas/fase1_obtencion.svg)

Las resoluciones son públicas y provienen del portal del STJ. Inicialmente se descargaron mediante su API, con un
máximo de 30 resoluciones por tipo penal, para equilibrar los delitos frecuentes y los escasos. **En octubre de 2026
la API comenzó a exigir verificación reCAPTCHA.** El descargador del proyecto (`descarga_datos/descargar.py`, para
cualquier materia) se detiene ante ella en el modo `api`; en el modo `navegador`, una persona la resuelve en una
ventana del portal y el script continúa con esa sesión. El proyecto nunca resuelve ni evade dicha verificación.

Desde entonces el corpus se amplía mediante **descarga manual** guiada por tres listas:

| Lista | Finalidad |
|---|---|
| `evaluacion/descargas/1_incompletas.csv` | Resoluciones extraídas cuando los PDF largos se muestreaban y se omitían páginas |
| `evaluacion/descargas/2_nuevas.csv` | Tipos penales con menos de 30 resoluciones en el corpus |
| `evaluacion/descargas/3_volver_a_bajar.csv` | Descargas que contenían otra resolución |

Cada PDF se nombra con la **fecha y hora de publicación al segundo**, la materia y el delito
(`2025-09-10-114715_penal_robo-calificado.pdf`); esa clave distingue dos resoluciones del mismo toca o del mismo día.

In [2]:
from copiloto_juridico.datos.procesamiento import parse_metadata_from_filename

for nombre in ["1_incompletas", "2_nuevas", "3_volver_a_bajar"]:
    df = pd.read_csv(f"evaluacion/descargas/{nombre}.csv", dtype=str)
    print(f"{nombre}.csv: {len(df):,} registros")
parse_metadata_from_filename("2025-09-10-114715_penal_robo-calificado.pdf")

1_incompletas.csv: 800 registros
2_nuevas.csv: 403 registros
3_volver_a_bajar.csv: 56 registros


{'fecha': '2025-09-10',
 'materia': 'penal',
 'delito_o_accion': 'robo-calificado',
 'tipo': 'penal/robo-calificado'}

## 4. Fase II. Procesamiento y depuración

![Fase II](diagramas/fase2_procesamiento.svg)

El comando `construir-corpus --pdfs CARPETA` procesa cada PDF **completo**; anteriormente se muestreaban 40 páginas de
los documentos de más de 50, con la consiguiente pérdida de contexto.

**Recorte estructural mediante expresiones regulares** (`extract_and_clean_legal_text`):

| Expresión | Patrón localizado | Decisión |
|---|---|---|
| `RE_STRUCTURAL_KEYWORDS_F1` | Encabezados de sección al inicio de línea: VISTOS, RESULTANDO, CONSIDERANDO…, también numerados («1.VISTOS») | El texto inicia en el primero de ellos solo si aparece dentro del primer 10 % del documento; en otro caso se conserva desde el principio |
| `RE_RESOLUTIVOS` | «resuelve», «proposiciones», «propositivas», «resolutivos», tolerando los espacios entre letras que introduce el PDF | Ubica los puntos resolutivos |
| `RE_FINAL_BOILERPLATE_F1` | Fórmulas de cierre: «Notifíquese», «Así lo resolvió», «Archívese» | El texto concluye en la primera fórmula **posterior** a los puntos resolutivos |

La última regla corrige un defecto medido sobre 1,889 PDF: un «Notifíquese» citado de la resolución de primera
instancia podía conservar apenas el 1 % del documento. A continuación, `limpiar_texto` suprime el anexo de datos
testados, los encabezados repetidos y las etiquetas de página.

**Validación de la descarga** (`contenido_en`): se toman 20 pasajes de 60 caracteres del texto que ya obraba en el
corpus y se verifica cuántos aparecen en el nuevo. Si aparece menos de la mitad, el PDF corresponde a otra
resolución (por ejemplo, el cumplimiento de una ejecutoria de amparo del mismo toca): se rechaza y se conserva el
texto anterior. **Resoluciones repetidas:** la misma resolución publicada dos veces con distinto nombre se conserva
una sola vez.

In [3]:
from copiloto_juridico.datos.procesamiento import extract_and_clean_legal_text, extract_text_from_pdf

pdfs = sorted(Path("data/raw").rglob("*.pdf"))
if pdfs:
    crudo = extract_text_from_pdf(pdfs[0])
    limpio = extract_and_clean_legal_text(crudo)
    print(f"{pdfs[0].name}: {len(crudo):,} caracteres extraídos → {len(limpio):,} tras el recorte estructural\n")
    print(limpio[:500], "…")
else:
    print("Los PDF son locales (data/raw/, fuera del repositorio); el texto depurado se encuentra en data/corpus/.")

2018-02-02-092953_penal_abuso-sexual-infantil-agravado-y-otro.pdf: 81,415 caracteres extraídos → 59,905 tras el recorte estructural

v i s t o s para resolver los autos del toca penal número 8922017 formado con motivo del recurso de apelación interpuesto por el agente del ministerio público, en contra del auto que dejó sin materia la probanza ofertada de su parte, de fecha 29 veintinueve de agosto de 2017 dos mil diecisiete, pronunciado por el juez segundo de lo penal del primer partido judicial , dentro de la causa criminal número 4732014-c, seguida en con tra de no.1eliminadoelnombrecompleto1, por su probable responsabilida …


In [4]:
from copiloto_juridico.datos.corpus import contenido_en

misma = "EL SUPREMO TRIBUNAL RESUELVE: PRIMERO. Se confirma el auto de vinculación a proceso. " * 5
otra = "Cumplimiento de ejecutoria de amparo: se deja insubsistente la resolución reclamada. " * 5
print(f"Misma resolución: {contenido_en(misma, misma):.2f} · otra resolución: {contenido_en(misma, otra):.2f}")
print(f"Descargas rechazadas en la construcción final: {len(pd.read_csv('data/corpus/descargas_rechazadas.csv'))}")
print(f"Resoluciones repetidas retiradas: {len(pd.read_csv('data/corpus/duplicados.csv'))}")

Misma resolución: 1.00 · otra resolución: 0.00
Descargas rechazadas en la construcción final: 54
Resoluciones repetidas retiradas: 8


## 5. Composición del corpus penal

Los 546 tipos penales que registra el portal se agrupan en **25 familias de delito** (`evaluacion/familias.py`)
mediante reglas sobre el nombre del tipo; la tabla completa, con la regla que decidió cada asignación, está en
`evaluacion/familias_penal.csv`. Ninguna asignación ha sido revisada por un especialista, lo que se declara como
limitación. La familia es la unidad de clasificación y de filtrado: agrupa variantes de un mismo delito (robo calificado,
robo equiparado) que un abogado consideraría comparables.

La familia **«cuestiones procesales»** reúne las resoluciones que no versan sobre un delito (quejas, excusas y
conflictos de competencia). La familia **«otros»** es residual: reúne los delitos que no encuadran en ninguna de las
demás (por ejemplo, aborto, allanamiento de morada, ataque peligroso o delitos contra el ambiente). En el análisis
significa «sin familia específica»; la búsqueda la trata como una familia más, de modo que una consulta sobre
allanamiento de morada se compara con ese grupo heterogéneo. En esos casos conviene seleccionar el delito o describir
los hechos con precisión.

In [5]:
from copiloto_juridico.datos.corpus import load_corpus
from copiloto_juridico.evaluacion.familias import cargar_familias

corpus = load_corpus()
penal = corpus[corpus["materia"] == "penal"]
familias = cargar_familias(penal["tipo"].unique().tolist())
originales = pd.read_csv("evaluacion/familias_penal.csv")["tipo"]
omitidas = pd.read_csv("evaluacion/paginas_omitidas.csv")
print(f"Resoluciones penales: {len(penal):,} · tipos del portal: {penal['tipo'].nunique()} "
      f"({penal['tipo'].isin(originales).groupby(penal['tipo']).first().sum()} en familias_penal.csv) · "
      f"familias: {len(set(familias.values()))}")
print(f"Publicadas entre {penal['fecha'].min()} y {penal['fecha'].max()}")
print(f"Resoluciones penales que aún tienen páginas omitidas: {omitidas['id'].isin(penal['id']).sum()}")
(penal.assign(familia=penal["tipo"].map(familias)).groupby("familia")
 .agg(resoluciones=("id", "size"), tipos=("tipo", "nunique")).sort_values("resoluciones", ascending=False))

Resoluciones penales: 2,495 · tipos del portal: 546 (546 en familias_penal.csv) · familias: 25
Publicadas entre 2018-02-02 y 2026-11-19
Resoluciones penales que aún tienen páginas omitidas: 54


,resoluciones,tipos
familia,,
homicidio,334,81
robo,301,63
delitos-sexuales,253,58
delitos-culposos,236,64
violencia-familiar,223,40
fraude-y-abuso-de-confianza,170,22
privacion-de-la-libertad-y-secuestro,134,26
lesiones,129,32
despojo,94,16


## 6. Fase III. Indexación híbrida

![Fase III](diagramas/fase3_indexacion.svg)

### 6.1 Fragmentación

Cada resolución se divide en fragmentos de **1,500 caracteres con traslape de 200**. El modelo de embeddings admite
como máximo 512 *tokens* (unidades de texto de 3 a 4 caracteres en promedio en español); 1,500 caracteres equivalen a
unos 400 tokens, de modo que ningún fragmento se trunca. El traslape evita que un razonamiento partido en la frontera
entre dos fragmentos se pierda. Las resoluciones penales se indexan **completas** (`MAX_CHUNKS_PENAL = 0`); con el
tope anterior de 12 fragmentos se indexaba alrededor del 40 % de cada sentencia.

### 6.2 Embeddings

Un *embedding* es la representación de un texto como un vector numérico (768 valores en este modelo), construido de
modo que los textos de significado semejante queden próximos aunque empleen palabras distintas. La proximidad se mide
con el **coseno** del ángulo entre vectores (1 = misma dirección). El modelo `multilingual-e5-base` exige anteponer
«query:» a la consulta y «passage:» a los fragmentos, y así se hace. Los embeddings se calculan una sola vez
(`indexar`), se guardan en `data/indice/` en precisión media (float16) y forman parte del repositorio, de modo que la
aplicación arranca sin recalcularlos.

### 6.3 Base vectorial ChromaDB

Al primer arranque, los embeddings se cargan en una base vectorial local **ChromaDB** con un índice **HNSW**
(*Hierarchical Navigable Small World*): un grafo que permite hallar en milisegundos los vecinos más próximos entre más
de 200 mil vectores sin compararlos todos, con distancia coseno y filtrado por metadatos (tipo de delito) [6]. Se
eligió porque funciona como biblioteca, sin servidor; persiste en disco (`data/chroma/`) y se reconstruye
automáticamente si cambia la huella del índice. La búsqueda semántica recupera los 300 fragmentos más próximos y los
agrupa por resolución (puntaje = coseno de su mejor fragmento).

### 6.4 Índice léxico BM25

En paralelo se construye, al arrancar, un índice **BM25** sobre el texto completo de cada resolución [7]: se cuentan
las palabras (en minúsculas y sin acentos, de 3 o más letras, sin palabras vacías ni «eliminado», «artículo» y
«fracción»), y cada término pesa más cuanto más aparece en la resolución y menos en el resto del corpus
(frecuencia inversa, IDF), con saturación de la frecuencia (k1 = 1.5) y normalización por longitud (b = 0.75). Su
función es complementaria: los embeddings pueden aproximar términos técnicos distintos («robo calificado» y «robo
equiparado», o dos medidas cautelares), mientras que BM25 **privilegia la coincidencia exacta** del nombre del
delito y de la figura procesal.

### 6.5 Fusión de rankings (RRF)

Las dos listas se combinan mediante *Reciprocal Rank Fusion* [10]: cada resolución suma 1 / (60 + posición) por cada
lista en que aparece. Se combinan **posiciones y no puntajes**, porque el coseno y el puntaje BM25 se expresan en
escalas no comparables; una resolución bien situada en ambas listas reúne dos evidencias independientes (semántica y
literal) y ocupa los primeros lugares. k = 60 es el valor propuesto por sus autores.

In [6]:
manifest = json.loads(Path("data/indice/manifest.json").read_text())
fragmentos = pd.read_parquet("data/indice/fragmentos.parquet")
fragmentos_penal = fragmentos["doc_id"].isin(penal["id"])
print(f"Modelo: {manifest['embedding_model']} · fragmentos de {manifest['chunk_size']} caracteres con traslape de "
      f"{manifest['chunk_overlap']} · {manifest['dim']} dimensiones")
print(f"Fragmentos de resoluciones penales: {fragmentos_penal.sum():,} "
      f"(mediana de {fragmentos[fragmentos_penal].groupby('doc_id').size().median():.0f} por resolución)")

Modelo: intfloat/multilingual-e5-base · fragmentos de 1500 caracteres con traslape de 200 · 768 dimensiones
Fragmentos de resoluciones penales: 186,971 (mediana de 51 por resolución)


In [7]:
from copiloto_juridico.clasificador import construir_clasificador
from copiloto_juridico.vectorstore import SentenciasStore

store = SentenciasStore(log=lambda _m: None)  # en el primer arranque construye ChromaDB; BM25 se arma en memoria
clasificador = construir_clasificador(store.corpus["tipo"].unique().tolist())
print("Índice listo.")

/home/nacho/Documents/repos/Proyecto_Final_CopilotoJurdicoHibridoconlocalLLMyRAG/Co-piloto-Jur-dico-H-brido-con-local-LLM-y-RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20954.67it/s]

Índice listo.


Demostración con la primera consulta de ejemplo de la aplicación: las cinco primeras resoluciones (por tipo penal)
según cada búsqueda y según su fusión.

In [8]:
from copiloto_juridico.ejemplos import ejemplos_de
from copiloto_juridico.embeddings import embed_query

ejemplo = ejemplos_de("abogado")[0]  # caso guía: el abogado de la víctima de un robo calificado
print(ejemplo.texto)
q = embed_query(ejemplo.texto)
tipo_de = store.corpus.set_index("id")["tipo"]
columnas = {}
for modo, titulo in [("densa", "semántica (embeddings)"), ("lexica", "literal (BM25)"), ("hibrida", "fusión RRF")]:
    res = store.buscar(ejemplo.texto, query_embedding=q, consulta_lexica=ejemplo.texto, modo=modo, num_docs=5)
    columnas[titulo] = [tipo_de[c.doc_id].split("/", 1)[1] for c in res]
pd.DataFrame(columnas, index=range(1, 6))

Mi cliente fue asaltado en una parada de camión: un hombre lo amenazó con un cuchillo y le quitó el celular. Lo reconoció en unas fotografías y el agresor fue vinculado a proceso, pero su defensa apeló el auto de vinculación. Quiero saber cómo ha resuelto la Sala apelaciones parecidas y qué valor le da al reconocimiento.


,semántica (embeddings),literal (BM25),fusión RRF
1,robo-calificado-y-tentativa-de-robo-calificado,violacion,homicidio-en-rina
2,corrupcion-de-menores,tentativa-de-homicidio-simple-intencional,contra-la-dignidad-de-las-personas
3,violencia-familiar-y-lesiones,homicidio-calificado-en-grado-de-tentativa,homicidio-simple
4,homicidio-en-rina,homicidio-simple-intencional-y-otro,violacion-y-corrupcion-de-menores
5,abuso-de-autoridad,atentados-al-pudor,abuso-de-autoridad


Con la consulta en bruto y sin filtro, ambas búsquedas recuperan también delitos ajenos (por ejemplo, violación o
feminicidio), arrastradas por términos como «mujer» o «víctima». Por ello el sistema no busca directamente con la
consulta: primero extrae los conceptos clave y clasifica la familia del delito, y busca dentro de ella (§ 7).

### 6.6 Parámetros del índice y su costo en memoria

El archivo `data/indice/manifest.json` registra los parámetros con que se construyó el índice; si alguno cambia, el
índice y la base ChromaDB se regeneran. Significado de cada campo:

| Campo | Significado | Razón de la decisión y efecto en memoria |
|---|---|---|
| `embedding_model` | Modelo que convierte texto en vectores | `multilingual-e5-base`: español, ~278 M parámetros; en GPU se carga en media precisión (float16) y ocupa del orden de medio GB, lo que permite convivir con el modelo de lenguaje; si la GPU está llena, se ejecuta en CPU |
| `chunk_size` | Longitud de cada fragmento, en **caracteres** | 1,500 caracteres ≈ 400 tokens < 512, el máximo del modelo: ningún fragmento se trunca |
| `chunk_overlap` | Caracteres compartidos entre fragmentos consecutivos | 200: un razonamiento partido en la frontera aparece completo en alguno de los dos fragmentos; aumenta el número de fragmentos en ~15 % |
| `num_docs` | Resoluciones indexadas | Solo resoluciones penales, indexadas completas |
| `num_chunks` | Total de fragmentos (vectores) | Determina el tamaño del índice: num_chunks × dim × bytes por número |
| `dim` | Dimensiones de cada vector | 768, fijado por el modelo; cada vector ocupa 768 × 2 bytes = 1.5 KB en float16 |

**Decisiones para que el sistema quepa en el equipo.** (i) Los embeddings se guardan en **float16** (media
precisión): la mitad de memoria que en float32, sin efecto apreciable en el orden por coseno. (ii) Los embeddings se
calculan **una sola vez** y se versionan en el repositorio; la aplicación no necesita GPU para indexar. (iii) El
índice BM25 es una **matriz dispersa** (solo se guardan los términos presentes en cada resolución) y descarta los
términos que aparecen en menos de 2 resoluciones o en más de la mitad. (iv) El modelo de lenguaje se usa
**cuantizado a 4 bits** (Q4_K_M) y con una ventana de 8,192 tokens, la necesaria para leer una resolución recortada.
(v) Nunca se cargan dos modelos de lenguaje a la vez.

La celda siguiente mide el consumo real en este equipo.

In [9]:
import psutil
import torch

from copiloto_juridico.embeddings import get_model

def gb(n):
    return f"{n / 1024 ** 3:.2f} GB"

def carpeta(ruta):
    return sum(f.stat().st_size for f in Path(ruta).rglob("*") if f.is_file())

emb = store.emb
modelo_emb = get_model()
parametros = sum(p.numel() for p in modelo_emb.parameters())
bm25 = store.bm25.matrix
filas = [
    ("Vectores del índice en memoria (float16)", f"{emb.shape[0]:,} × {emb.shape[1]}", gb(emb.nbytes)),
    ("Los mismos vectores en float32 (no usado)", "", gb(emb.shape[0] * emb.shape[1] * 4)),
    ("data/indice/ en disco (versionado en git)", "", gb(carpeta("data/indice"))),
    ("data/chroma/ en disco (base HNSW, local)", "", gb(carpeta(config.CHROMA_DIR))),
    ("Matriz BM25 dispersa", f"{bm25.shape[0]:,} resoluciones × {bm25.shape[1]:,} términos, {bm25.nnz:,} no nulos",
     gb(bm25.data.nbytes + bm25.indices.nbytes + bm25.indptr.nbytes)),
    ("Modelo de embeddings", f"{parametros / 1e6:.0f} M parámetros en {modelo_emb.device}, {next(modelo_emb.parameters()).dtype}",
     gb(sum(p.numel() * p.element_size() for p in modelo_emb.parameters()))),
    ("Memoria RAM de este proceso (índice + BM25 + modelo)", "", gb(psutil.Process().memory_info().rss)),
    ("RAM total del equipo", "", gb(psutil.virtual_memory().total)),
]
if torch.cuda.is_available():
    filas.append(("GPU", torch.cuda.get_device_name(0), gb(torch.cuda.get_device_properties(0).total_memory)))
pd.DataFrame(filas, columns=["elemento", "detalle", "tamaño"]).set_index("elemento")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17749.42it/s]

GPU sin memoria para el modelo de embeddings; se usa la CPU.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16674.65it/s]

,detalle,tamaño
elemento,,
Vectores del índice en memoria (float16),"186,971 × 768",0.27 GB
Los mismos vectores en float32 (no usado),,0.53 GB
data/indice/ en disco (versionado en git),,0.27 GB
"data/chroma/ en disco (base HNSW, local)",,0.76 GB
Matriz BM25 dispersa,"2,495 resoluciones × 78,874 términos, 3,621,581 no nulos",0.04 GB
Modelo de embeddings,"278 M parámetros en cpu, torch.float32",1.04 GB
Memoria RAM de este proceso (índice + BM25 + modelo),,4.31 GB
RAM total del equipo,,30.64 GB
GPU,NVIDIA GeForce RTX 4060 Laptop GPU,7.62 GB


## 7. Fase IV. Flujo de análisis de una consulta

![Fase IV](diagramas/fase4_flujo.svg)

**Paso 1. Conceptos clave.** El modelo de lenguaje extrae de 1 a 3 conceptos jurídicos: primero el delito con su
modalidad, deducido de los hechos cuando no se nombra, y después el problema procesal (vinculación a proceso,
prisión preventiva, legalidad de la detención). Si no logra extraerlos, se emplea la consulta íntegra.

**Paso 2. Clasificación híbrida.** La consulta se compara con cada una de las 25 familias mediante cuatro señales:
similitud semántica con la descripción de la familia (peso 0.20); coincidencia de lemas entre los conceptos y la
familia, medida con el índice de Jaccard (0.40); concisión de la categoría (0.05), y el **voto** de las resoluciones
más parecidas, es decir, la familia a la que pertenecen, ponderada por su posición RRF (0.35). La señal KEM se
calcula pero no pondera, por coincidir con Jaccard (correlación 0.995). **La lematización (reducción de cada palabra a
su forma base con spaCy: «robaron» → «robar») se emplea únicamente aquí**, para comparar palabras de la consulta y de la
familia; ni los embeddings ni BM25 lematizan, y la fragmentación responde al límite de 512 tokens del modelo de
embeddings, no a la lematización. Cada familia se describe con los nombres de
sus tipos más generales; con ello la familia correcta queda primera en 13 de 14 ejemplos penales (antes, 12;
regresiones `20261112-180034_139bfe6` y `20261112-180217_99bb183`).

**Paso 3. Búsqueda adaptativa.** Se buscan resoluciones de las 3 familias más probables o de los delitos que
seleccionó el usuario; si el filtro arroja pocas, se completa sin filtro y así se indica. Las candidatas se ordenan
primero por coincidencia de familia y de acto impugnado (leído por reglas): en los ejemplos penales, de los 3
primeros candidatos, 37 de 42 son de la familia correcta, frente a 32 de 42 con el orden RRF (regresiones
`20261113-000836_137c238` y `20261113-001214_a5c476b`).

**Paso 4. Análisis de precedentes.** Para cada candidata, en el orden por familia y acto: **QC_1** (opcional,
apagado por defecto: sobre 39 consultas de desarrollo no cambió los 3 precedentes mostrados, P@3 +0.000
[−0.034, +0.034], y costaba 3 de las 5 llamadas al modelo; `20261124-014247_0f2894a_precedentes`), el modelo lee la ficha precalculada de la candidata (delito, acto
impugnado, agravios, sentido) y decide si trata el mismo delito o familia o el mismo problema procesal (descarta el 14 % de los
candidatos, frente al 55 % si se exige «mismo tema y contexto» leyendo el inicio; corrida `20261112-184123_2c7be47`; leer la ficha en lugar del fragmento
sube su recall de 0.725 a 0.844 en desarrollo, `20261123-230824_5fc9b67_qc`); **ficha**, el modelo devuelve
un objeto JSON de 14 campos extraídos solo del texto, con reintento si el formato es inválido; **verificaciones
automáticas**, el sentido del fallo se contrasta con los puntos resolutivos leídos por reglas, cada artículo citado se
busca en la resolución y **QC_2** comprueba que el delito de la ficha pertenezca a una familia buscada. Las fichas se
calculan previamente (con el modelo principal y, para las faltantes, con el de respaldo) y la aplicación reutiliza la
vigente. Si una resolución excede la ventana del modelo
(8,192 tokens, unos 20,000 caracteres), se envían su inicio, el estudio de los agravios y los puntos resolutivos; no
se resume.

**Paso 5. Síntesis.** Por reglas se determina la parte favorecida por cada precedente, el acusado o la acusación
(quién apeló y sentido del fallo). El modelo redacta la respuesta **exclusivamente con las fichas**, cita cada
afirmación ([P1], [P2]…) y el código verifica que la cita exista y concuerde con la ficha; no pronostica el resultado. Agregar al prompt un ejemplo de líneas citadas no mejoró las
citas de forma significativa y, con contenido jurídico, el modelo lo copiaba (`20261124-003427_905c255_sintesis-citas`).

**Delito seleccionado por el usuario (opcional).** La aplicación sugiere, sin invocar al modelo de lenguaje, las
familias que deduce de la descripción. La sugerencia contiene la familia correcta en 73 de 100 consultas de
desarrollo (81 con los conceptos del modelo de lenguaje) y en 13 de 14 ejemplos; por ello es una orientación y no un
filtro automático (`evaluacion/errores/sugerencia_delito.md`).

In [10]:
from copiloto_juridico.evaluacion.familias import etiqueta_familia
from copiloto_juridico.pipeline import sugerir_familias

sugerencia = sugerir_familias(store, clasificador, ejemplo.texto)
print("Sugerencia inicial de delito:", ", ".join(map(etiqueta_familia, sugerencia)))
esperada = ejemplo.tipo_corpus.split("/")[1].split("-")[0]  # "robo" para el caso guía
print("La sugerencia contiene la familia del caso guía." if any(f.startswith(esperada) for f in sugerencia) else
      "La sugerencia no contiene la familia del caso guía; el análisis completo usa además los conceptos del modelo.")

Sugerencia inicial de delito: homicidio, robo, delitos sexuales
La sugerencia contiene la familia del caso guía.


In [11]:
from copiloto_juridico.llm import get_client
from copiloto_juridico.pipeline import Copiloto, Opciones

cliente = get_client(llm)
disponible, mensaje = cliente.health()
resultado = None
if not disponible:
    print(f"Modelo de lenguaje no disponible ({mensaje}); se omite el análisis completo.")
else:
    # Sin delito seleccionado: el sistema lo determina. Para seleccionarlo: Opciones(familias=("robo",))
    for ev in Copiloto(store, clasificador, cliente).analizar(ejemplo.texto, Opciones()):
        if ev.tipo == "paso_fin":
            print(f"✓ {ev.paso}: {ev.mensaje or ''}")
        elif ev.tipo == "error":
            print(f"✗ {ev.paso}: {ev.mensaje}")
        elif ev.tipo == "resultado":
            resultado = ev.datos["resultado"]

✓ entidades: robo calificado, vinculación a proceso, apelación del auto de vinculación


✓ clasificacion: PENAL · familia: robo
✓ busqueda: 30 candidatos · Filtro estricto


✓ analisis: 3 informes válidos · 0 descartados


✓ sintesis: 


In [12]:
if resultado is not None:
    print(f"Conceptos clave: {resultado.entidades}")
    display(resultado.tabla_clasificacion[["categoría", "score_híbrido", "coseno", "jaccard", "voto_recuperación"]].head(5))

Conceptos clave: robo calificado, vinculación a proceso, apelación del auto de vinculación


,categoría,score_híbrido,coseno,jaccard,voto_recuperación
0,PENAL · familia: robo,0.506943,0.826997,0.166667,0.766248
1,PENAL · familia: homicidio,0.400000,0.812812,0.000000,1.000000
2,PENAL · familia: delitos sexuales,0.299964,0.821287,0.000000,0.547614
3,PENAL · familia: contra la dignidad de las personas,0.263598,0.836048,0.000000,0.080693
4,PENAL · familia: privacion de la libertad y secuestro,0.251407,0.831401,0.000000,0.178696


**Consumo del modelo de lenguaje.** Ollama informa, con el modelo cargado, cuánto ocupa en total y cuánto en la
GPU; si no cabe, la parte restante se ejecuta desde la RAM, con una pérdida considerable de velocidad.

In [13]:
import requests

for m in requests.get(f"{llm.base_url}/api/ps", timeout=5).json().get("models", []):
    print(f"{m['name']}: {m['details']['parameter_size']} parámetros, cuantización {m['details']['quantization_level']}"
          f" · ocupa {gb(m['size'])}, de los cuales {gb(m['size_vram'])} en la GPU")

llama3.1:8b: 8.0B parámetros, cuantización Q4_K_M · ocupa 6.57 GB, de los cuales 5.85 GB en la GPU


La misma búsqueda híbrida, ahora con los conceptos clave del paso 1 y el filtro de las 3 familias del paso 2:

In [14]:
if resultado is not None:
    filtro = [t for c in resultado.tabla_clasificacion["tipo"].head(config.NUM_CATEGORIAS_FILTRO)
              for t in clasificador.tipos_de(c)]
    res = store.buscar(ejemplo.texto, tipos=filtro, query_embedding=q,
                       consulta_lexica=f"{ejemplo.texto} {resultado.entidades}", num_docs=5)
    display(pd.DataFrame({"fusión RRF con conceptos y filtro": [tipo_de[c.doc_id].split("/", 1)[1] for c in res]},
                         index=range(1, 6)))

,fusión RRF con conceptos y filtro
1,homicidio-simple
2,homicidio-en-rina
3,corrupcion-de-menores
4,violacion-y-corrupcion-de-menores
5,homicidio-simple-intencional-y-robo-calificado


In [15]:
Markdown(resultado.markdown()) if resultado is not None else print("Sin resultado.")

# Informe del Co-piloto Jurídico

**Consulta:** Mi cliente fue asaltado en una parada de camión: un hombre lo amenazó con un cuchillo y le quitó el celular. Lo reconoció en unas fotografías y el agresor fue vinculado a proceso, pero su defensa apeló el auto de vinculación. Quiero saber cómo ha resuelto la Sala apelaciones parecidas y qué valor le da al reconocimiento.

**Categoría detectada:** PENAL · familia: robo  
**Búsqueda:** Filtro estricto  
**Modelo:** llama3.1:8b · **Tiempo:** 32 s · **Veredictos ambiguos:** 0

---
## Síntesis de precedentes relevantes

### Resultado de cada precedente
- **Precedente 1:** apeló defensa contra auto de vinculación a proceso; la Sala CONFIRMA → favorece a **la acusación (Ministerio Público o víctima)**.
- **Precedente 2:** apeló defensa del imputado y ministerio público contra auto de vinculación a proceso; la Sala CONFIRMA → parte favorecida no determinada.
- **Precedente 3:** apeló agente del ministerio público contra auto de vinculación a proceso dictado por el [eliminado], enjuiciamiento y justicia integral para adolescentes, adscrito al centro de justicia penal del [eliminado] judicial con sede en [eliminado]; la Sala MODIFICA → favorece a **en parte, la acusación (Ministerio Público o víctima)**.

### Respuesta breve
La Sala Penal del STJ Jalisco ha considerado que la valoración de los datos probatorios fue conjunta e integral en un caso de robo calificado [P1], y también ha confirmado autos de vinculación a proceso cuando se han presentado pruebas suficientes para establecer la participación del imputado en el delito, como es el caso del reconocimiento en cámara de gesell [P2]. Además, la Sala ha considerado que existen datos de prueba suficientes que acreditan la utilización del cuchillo en un robo calificado [P3].

### Argumentos que la Sala aceptó o rechazó
* La defensa argumentó que los testigos no son objetivos y tienen interés en favorecer al imputado, pero la Sala considera que el tribunal comparte el criterio del juez de control sobre la falta de objetividad de los testigos [P1].
* El imputado argumentó que la detención fue ilegal y que se obtuvieron pruebas violentando los artículos 14 y 16 de la Constitución, pero la Sala considera que el imputado no aportó suficientes pruebas para demostrar la ilegalidad de la detención [P2].
* El Ministerio Público argumentó insuficiencia probatoria para la agravante contenida en la fracción xi del artículo 236 del Código Penal del Estado, pero la Sala considera que existen datos de prueba suficientes que acreditan la utilización del cuchillo [P3].

### Qué revisar en las sentencias
Es recomendable revisar los precedentes completos para comprender el contexto y la fundamentación legal de cada caso. En particular, es importante leer el Precedente 1 para entender cómo se valoró la falta de objetividad de los testigos [P1], y el Precedente 3 para ver cómo se consideraron suficientes las pruebas para acreditar la utilización del cuchillo en un robo calificado [P3].

---
## Apéndice: análisis detallado de precedentes

### Precedente 1: PENAL · robo calificado

| Documento | Fecha | Materia | Similitud | Fuente |
|---|---|---|---|---|
| `2025-04-24-105911_penal_robo-calificado.pdf` | 2025-04-24 | PENAL | 0.860 | Filtro estricto |

#### Resumen
Se confirma el auto de vinculación a proceso por el delito de robo calificado. La defensa apeló la decisión del juez de control, pero la Sala considera que la valoración de los datos probatorios fue conjunta e integral.

#### Datos del proceso
- **Tribunal:** Sala Penal · [DATO NO DISPONIBLE]
- **Toca / expediente:** Toca penal 3742024
- **Acto impugnado:** Auto de vinculación a proceso · **Apelante:** Defensa
- **Delito(s):** Delito de robo calificado, previsto por el artículo 233 en relación al 236 fracción xi del código penal del estado de Jalisco
- **Sentido del fallo:** CONFIRMA
- **Parte favorecida:** la acusación (Ministerio Público o víctima)

#### Agravios y su calificación
- La Sala considera que los testigos de la defensa no son objetivos y tienen interés en favorecer al imputado. — **INFUNDADO**: Porque el tribunal comparte el criterio del juez de control sobre la falta de objetividad de los testigos.
- La Sala considera que la decisión del a quo es válida y razonable. — **INFUNDADO**: Porque el tribunal considera que la valoración de los datos probatorios fue conjunta e integral.

#### Hechos y decisión
**Hechos:** El imputado fue vinculado a proceso por el delito de robo calificado. La defensa apeló la decisión del juez de control.

**Decisión:** Se confirma el auto de vinculación a proceso dictado por el juez de control.

#### Fundamento legal
- Artículo 233 en relación al 236 fracción xi del código penal del estado de Jalisco
- Artículo 461 del Código Nacional de Procedimientos Penales

#### Pruebas decisivas
- El informe policial homologado
- La declaración ministerial del imputado

### Precedente 2: PENAL · robo calificado y privacion ilegal de la libertad y otros derechos

| Documento | Fecha | Materia | Similitud | Fuente |
|---|---|---|---|---|
| `2023-11-23-134518_penal_robo-calificado-y-privacion-ilegal-de-la-libertad-y-otros-derechos.pdf` | 2023-11-23 | PENAL | 0.865 | Filtro estricto |

#### Resumen
Se confirma el auto de vinculación a proceso dictado en contra del imputado por el delito de robo calificado y delito de privación ilegal de la libertad. El imputado fue detenido junto a otros dos sujetos en posesión de droga y se realizó un reconocimiento en cámara de gesell, pero el imputado argumenta que este es viciado por no cumplir con las formalidades de ley.

#### Datos del proceso
- **Tribunal:** Sala Penal del Primer Distrito Judicial · 09 de febrero de 2023
- **Toca / expediente:** Toca penal J.O.3312022
- **Acto impugnado:** Auto de vinculación a proceso · **Apelante:** Defensa del imputado y Ministerio Público
- **Delito(s):** Delito de robo calificado (artículo 233, en contexto con el 236, fracciones ix y xi, del código penal para el estado de Jalisco) y delito de privación ilegal de la libertad (artículo 193 del citado cuerpo de leyes)
- **Sentido del fallo:** CONFIRMA
- **Parte favorecida:** [DATO NO DISPONIBLE]

#### Agravios y su calificación
- El imputado argumenta que la detención fue ilegal y que se obtuvieron pruebas violentando los artículos 14 y 16 de la Constitución. — **INFUNDADO**: La Sala considera que el imputado no aportó suficientes pruebas para demostrar la ilegalidad de la detención.
- El imputado argumenta que el reconocimiento en cámara de gesell es viciado. — **INFUNDADO**: La Sala considera que el imputado no aportó suficientes pruebas para demostrar la viciación del reconocimiento.
- El imputado argumenta que no es posible que las mismas personas cometieran el robo y siguieron tripulando el mismo vehículo. — **INFUNDADO**: La Sala considera que este argumento es infundado porque no hay suficientes pruebas para demostrar la imposibilidad de que las mismas personas cometieran el robo y siguieron tripulando el mismo vehículo.
- El imputado argumenta que el solo hecho de haber una identificación en contra no exime al agente del Ministerio Público de allegar más pruebas. — **INFUNDADO**: La Sala considera que este argumento es infundado porque la identificación en contra ya es suficiente para establecer la participación del imputado en el delito.

#### Hechos y decisión
**Hechos:** El imputado fue detenido junto a otros dos sujetos en posesión de droga. Se realizó un reconocimiento en cámara de gesell, pero el imputado argumenta que este es viciado por no cumplir con las formalidades de ley.

**Decisión:** Se confirma el auto de vinculación a proceso dictado en contra del imputado por el delito de robo calificado y delito de privación ilegal de la libertad.

#### Fundamento legal
- Artículo 233, en contexto con el 236, fracciones ix y xi, del código penal para el estado de Jalisco
- Artículo 193 del citado cuerpo de leyes

#### Pruebas decisivas
- El reconocimiento en cámara de gesell
- La identificación en contra del imputado

### Precedente 3: PENAL · robo simple

| Documento | Fecha | Materia | Similitud | Fuente |
|---|---|---|---|---|
| `2026-11-10-131731_penal_robo-simple.pdf` | 2026-11-10 | PENAL | 0.859 | Filtro estricto |

#### Resumen
El Ministerio Público apeló la vinculación a proceso por robo simple, argumentando insuficiencia probatoria para la agravante. La Sala modificó la resolución, considerando suficientes los datos de prueba.

#### Datos del proceso
- **Tribunal:** Primera Sala del Tribunal de Alzada · 13 de marzo de 2026
- **Toca / expediente:** Toca penal digital número 2022026
- **Acto impugnado:** Auto de vinculación a proceso dictado por el [eliminado], enjuiciamiento y justicia integral para adolescentes, adscrito al centro de justicia penal del [eliminado] judicial con sede en [eliminado] · **Apelante:** Agente del Ministerio Público
- **Delito(s):** Robo calificado (artículo 233 en relación al 236 fracción xi, Código Penal del Estado)
- **Sentido del fallo:** MODIFICA
- **Parte favorecida:** en parte, la acusación (Ministerio Público o víctima)

#### Agravios y su calificación
- Insuficiencia probatoria para la agravante contenida en la fracción xi del artículo 236 del Código Penal del Estado — **FUNDADO**: Existen datos de prueba suficientes que acreditan la utilización del cuchillo.

#### Hechos y decisión
**Hechos:** El imputado ingresó a una tienda de conveniencia, tomó un producto sin pagar y posteriormente amenazó con un cuchillo a la víctima.; La víctima entregó dinero en efectivo por miedo a que el imputado lo agrediera.; Se encontraron pruebas como el cuchillo y fotografías del lugar de los hechos.

**Decisión:** Se modifica el auto de vinculación a proceso para incluir la agravante y se decreta auto de vinculación a proceso por robo calificado.

#### Fundamento legal
- Artículo 461 del Código Nacional de Procedimientos Penales
- Artículo 233 en relación al 236 fracción xi, Código Penal del Estado

#### Pruebas decisivas
- Narrativa de los primeros respondientes
- Inspección, aseguramiento y cadena de custodia de los objetos
- Cuchillo encontrado en el suelo


---
_Herramienta de apoyo basada en precedentes públicos del STJ Jalisco. No sustituye la asesoría de un profesional del derecho._

## 8. Catálogo de instrucciones al modelo (*prompts*)

Las instrucciones se dividen en dos grupos: las que **opera la aplicación** en cada consulta y las que se emplean
**solo en la evaluación**. Todas residen en `src/copiloto_juridico/prompts.py`; las de la línea base se
conservan únicamente para que la evaluación las compare, mediante variables de entorno.

**Integración de la recuperación con las instrucciones.** El paso 1 recibe solo la consulta. QC_1 recibe la consulta
y la ficha precalculada de la resolución recuperada (o, sin ficha, el fragmento más parecido). La ficha recibe el texto de la resolución recuperada, recortado a la ventana
del modelo; esta es la parte de *generación aumentada por recuperación*. La síntesis recibe solo las fichas, nunca el
texto completo ni conocimiento externo, y debe citar cada afirmación.

In [16]:
from copiloto_juridico import prompts

catalogo = [
    ("Aplicación", "1. Conceptos clave", "PROMPT_EXTRACCION_ENTIDADES_V11", "consulta → 1 a 3 conceptos"),
    ("Aplicación", "4. QC_1", "PROMPT_VERIFICACION_RELEVANCIA_V11", "consulta + fragmento → razonamiento y VEREDICTO SI/NO"),
    ("Aplicación", "4. Ficha", "PROMPT_FICHA_PENAL_JSON", "resolución recortada → JSON de 14 campos"),
    ("Aplicación", "4. Ficha (tercer intento)", "PROMPT_FICHA_FALTANTES", "resolución → solo los campos obligatorios faltantes"),
    ("Aplicación", "5. Síntesis", "PROMPT_SINTESIS_FINAL_V11", "consulta + fichas → respuesta con citas [Pn]"),
    ("Evaluación", "Juez: resolución", "PROMPT_JUEZ_SENTENCIA", "inicio + resolutivos → familia, acto, apelante, sentido"),
    ("Evaluación", "Juez: relevancia", "PROMPT_JUEZ_RELEVANCIA", "consulta + resolución → relevancia 0/1/2"),
    ("Evaluación", "Juez: consulta", "PROMPT_JUEZ_CONSULTA", "consulta de prueba → ¿es válida?"),
    ("Evaluación", "Consultas de prueba", "PROMPT_BORRADOR_CONSULTA", "resolución → consulta redactada como defensor"),
]
pd.DataFrame([{"uso": u, "paso": p, "prompt": n, "entrada → salida": e, "caracteres": len(getattr(prompts, n))}
              for u, p, n, e in catalogo])

,uso,paso,prompt,entrada → salida,caracteres
0,Aplicación,1. Conceptos clave,PROMPT_EXTRACCION_ENTIDADES_V11,consulta → 1 a 3 conceptos,841
1,Aplicación,4. QC_1,PROMPT_VERIFICACION_RELEVANCIA_V11,consulta + fragmento → razonamiento y VEREDICTO SI/NO,1152
2,Aplicación,4. Ficha,PROMPT_FICHA_PENAL_JSON,resolución recortada → JSON de 14 campos,1793
3,Aplicación,4. Ficha (tercer intento),PROMPT_FICHA_FALTANTES,resolución → solo los campos obligatorios faltantes,348
4,Aplicación,5. Síntesis,PROMPT_SINTESIS_FINAL_V11,consulta + fichas → respuesta con citas [Pn],1778
5,Evaluación,Juez: resolución,PROMPT_JUEZ_SENTENCIA,"inicio + resolutivos → familia, acto, apelante, sentido",1652
6,Evaluación,Juez: relevancia,PROMPT_JUEZ_RELEVANCIA,consulta + resolución → relevancia 0/1/2,1353
7,Evaluación,Juez: consulta,PROMPT_JUEZ_CONSULTA,consulta de prueba → ¿es válida?,641
8,Evaluación,Consultas de prueba,PROMPT_BORRADOR_CONSULTA,resolución → consulta redactada como defensor,966


In [17]:
print(prompts.PROMPT_FICHA_PENAL_JSON)


Eres un Analista Legal Penal que apoya a la DEFENSA en segunda instancia. Lee la SENTENCIA PENAL (resolución de un
recurso de apelación de una Sala Penal) y devuelve su ficha como UN objeto JSON con exactamente estas claves:

{{
  "tribunal": "Sala que resolvió",
  "fecha_resolucion": "fecha exacta de la resolución",
  "identificador": "toca / expediente / causa tal como aparece",
  "etapa_procesal": "etapa del proceso en que se dictó la resolución apelada",
  "acto_impugnado": "qué resolución se apeló (auto de vinculación a proceso, prisión preventiva, sentencia definitiva de juicio oral, sobreseimiento, otro)",
  "apelante": "quién apeló: defensa, Ministerio Público, asesor jurídico de la víctima",
  "delitos": "delito(s) imputados o acreditados, con su modalidad",
  "hechos_clave": "2 a 4 frases con la conducta y sus circunstancias de modo, tiempo y lugar",
  "agravios": [{{"agravio": "argumento del apelante, en una frase", "calificacion": "FUNDADO | INFUNDADO | INOPERANTE | NO DIS

## 9. Fase V. Evaluación

![Fase V](diagramas/fase5_evaluacion.svg)

**Referencia sin anotadores humanos.** No se contó con abogados para anotar resultados; la referencia se construyó
de forma automática y externa al sistema:

- **Delito:** la categoría que el propio STJ asignó a cada resolución en su portal (etiqueta externa), llevada a su
  familia.
- **Acto impugnado, apelante y sentido del fallo:** mayoría absoluta entre las reglas aplicadas al texto
  (`metadatos.py`) y el juez; sin mayoría, el dato no se computa.
- **Relevancia de un par (consulta, resolución):** 2 si coinciden delito y problema procesal; 1 si coincide uno; 0 si
  ninguno.

**Validación de las etiquetas.** La etiqueta externa (categoría del STJ) sirve para validar al juez: un juez solo se
admite si identifica la familia del delito con exactitud de al menos **0.80**, umbral fijado antes de ejecutarlo.
Las etiquetas internas (reglas) se contrastan con el juez mediante el coeficiente kappa de Cohen [11], que descuenta
el acuerdo esperado por azar.

**Jueces de otra familia de modelos, y dos de ellos.** Un modelo de lenguaje tiende a favorecer sus propias
respuestas (sesgo de autopreferencia) [8][9]; por ello los jueces no pertenecen a la familia Llama de la aplicación.
Se emplearon dos, `gemma2:9b` y `mistral:7b`, para no depender de un solo lector. Con el corpus anterior, solo
`gemma2:9b` acreditó el umbral (0.845, n = 401). Con el corpus final y las 25 familias, su exactitud es 0.742, IC 95 % [0.709, 0.773]
(n = 725, `20261121-031724_723fe6b_referencias`), por lo que, conforme al criterio fijado de antemano, se mantiene como lector de
contraste y la referencia se apoya en la etiqueta del STJ y en las reglas. Sus diferencias se concentran en los delitos
culposos, en la violación cometida en el ámbito familiar y en la familia residual «otros»
(`evaluacion/errores/juez_familia.md`).

**Consecuencia para las métricas.** Las métricas principales dependen solo de la etiqueta externa (la categoría del
STJ). Las de acto impugnado, apelante y sentido se miden contra las reglas, que la aplicación también utiliza para
ordenar candidatos, lo que se indica junto a esos resultados.

**Muestra estable.** Juzgar todo el corpus con un juez tomaba alrededor de 7 horas. Se utiliza una muestra
determinada por la huella (*hash*) del identificador con semilla fija: 30 % de las resoluciones y 50 % de las
consultas, con todos sus pares. Cada elemento queda siempre dentro o siempre fuera, aunque el corpus crezca.

**Precisión del fundamento legal.** Sin anotadores, la referencia son los artículos que la propia Sala invoca como
fundamento: los que siguen a «con fundamento en», «de conformidad con», «con apoyo en», «en términos de» o «con base
en», y los de los puntos resolutivos. La **precisión** mide qué fracción de los artículos que cita la ficha figura en
ese fundamento; la exhaustividad es baja por construcción, porque la ficha resume los artículos principales. Además, cada
artículo de la ficha se busca en todo el texto y, si no aparece, la aplicación lo marca con «verificar».

**Congelamiento.** `congelar` registra la huella del código, los datos, las instrucciones y el modelo; cada corrida
de la partición de prueba se niega a ejecutarse si algo cambió y solo se permite una vez.

In [18]:
from copiloto_juridico.evaluacion.reporte import ultimo_resultado

ref = ultimo_resultado("referencias")
print(f"Corrida: {ref['_archivo']}")
for nombre in ["Validación de los jueces", "Acuerdo entre lectores de la sentencia", "Cobertura de la referencia"]:
    t = ref["tablas"][nombre]
    display(Markdown(f"**{nombre}.** {t['descripcion']}"))
    display(pd.DataFrame(t["filas"], columns=t["columnas"]).round(3))

Corrida: 20261121-031724_723fe6b_referencias.json


**Validación de los jueces.** Exactitud de la familia del delito que lee cada juez contra la categoría que asignó el STJ (referencia externa; una respuesta inválida cuenta como error). Un juez se usa si alcanza el mínimo, fijado antes de correrlos.

,juez,sentencias,con respuesta válida,exactitud familia (STJ),IC95,mínimo,se usa
0,gemma2:9b,725,725,0.742,"[0.709, 0.773]",0.8,no


**Acuerdo entre lectores de la sentencia.** Reglas y jueces leen acto impugnado, apelante y sentido; kappa de Cohen sobre las sentencias donde ambos respondieron.

,campo,lector A,lector B,sentencias,acuerdo,kappa
0,acto_impugnado,reglas,juez:gemma2:9b,583,0.650,0.585
1,apelante,reglas,juez:gemma2:9b,567,0.852,0.790
2,sentido,reglas,juez:gemma2:9b,557,0.846,0.770


**Cobertura de la referencia.** Sentencias con mayoría absoluta entre reglas y jueces usables (ninguno); sin mayoría no cuentan.

,campo,sentencias,con referencia (mayoría)
0,acto_impugnado,725,583
1,apelante,725,568
2,sentido,725,561


### 9.1 Guía de interpretación de las métricas

Valores de la corrida final (partición de prueba, congelado `a15b83c852281010`, 46 consultas validadas).

| Métrica | Pregunta que responde | Cálculo | Resultado y lectura |
|---|---|---|---|
| **Hit@3 de familia** (`recuperacion`) | ¿El filtro de búsqueda contiene el delito correcto? | Fracción de consultas cuya familia del STJ está entre las 3 que elige el clasificador; la sentencia de origen no vota (*leave-one-out*); IC 95 % de Wilson | 0.717 [0.575, 0.827]: en 7 de cada 10 consultas el sistema busca en la familia correcta; la línea base k-NN, 0.435 |
| **Hit@k** | ¿Aparece la sentencia de la que se redactó la consulta? | Fracción de consultas con esa sentencia entre los k primeros | Hit@5 de la app 0.174; con re-ranker 0.370. Mide un solo documento: hay otros precedentes útiles |
| **MRR** | ¿Qué tan arriba aparece esa sentencia? | Promedio de 1/posición (1 = primera, 0.5 = segunda; lista de 30) | App 0.152; BM25 solo 0.233 |
| **nDCG@5** | ¿Los 5 primeros están bien ordenados por utilidad? | Ganancia con descuento por posición, normalizada por el orden ideal; relevancia 2 = mismo delito y mismo problema procesal, 1 = uno de los dos | App 0.285; BM25 0.243; embeddings 0.304; 1 = orden perfecto |
| **P@5** | ¿Cuántos de los 5 primeros son pertinentes? | Fracción del top 5 con relevancia ≥ 1 | App 0.265: 1.3 de los 5 primeros |
| **Diferencia pareada** | ¿Una variante supera a otra? | Media por consulta de (A − B), IC 95 % por *bootstrap* pareado (1,000 remuestreos); efecto mínimo detectable con alfa 5 % y potencia 80 % | Significativa si el intervalo no incluye 0: el híbrido supera a k-NN en Hit@3 (+0.283); el re-ranker mejora Hit@5 (+0.196) pero no nDCG@5 |
| **Exactitud de la ficha** (`fichas-referencia`) | ¿La ficha lee bien delito, acto, apelante y sentido? | Coincidencia con la referencia por campo; IC de Wilson | Familia 0.917, acto 0.883, apelante 0.830, sentido 0.722 |
| **Precisión del fundamento legal** | ¿Los artículos que cita la ficha son los que invocó la Sala? | Por ficha, citados que figuran en el fundamento ÷ citados; promedio con IC *bootstrap* | 0.644 [0.600, 0.690]: casi 2 de cada 3 artículos citados son fundamento de la Sala |
| **Exhaustividad del fundamento** | ¿Cuánto del fundamento recoge la ficha? | Citados en el fundamento ÷ artículos del fundamento | 0.133: la ficha resume los principales; la sentencia invoca muchos |
| **Artículos no encontrados** | ¿La ficha cita artículos ausentes de la sentencia? | Artículos de la ficha que no aparecen en el texto | 7 de 350 fichas; la app los marca con «verificar» |
| **Precisión / recall / kappa de QC_1** (`qc`) | ¿El control de relevancia descarta precedentes útiles? | Veredictos contra la referencia en 776 pares; kappa descuenta el acuerdo por azar | Recall 0.721: conserva 7 de cada 10 relevantes; precisión 0.322 y kappa 0.051: deja pasar muchos no relevantes: está diseñado para no descartar; el orden por facetas y QC_2 actúan después |
| **Esfuerzo** (`esfuerzo`) | ¿Cuántas sentencias hay que abrir para hallar 3 útiles? | Sentencias abiertas hasta reunir 3 útiles (misma familia y mismo acto impugnado), tope 30 (31 = no llegó); 9 consultas | Con el delito elegido, mediana 22 frente a 31 del portal; elegir el delito ahorra 7.0 [+1.8, +13.2] |
| **Coherencia de familias** (`familias --coherencia`) | ¿Los tipos agrupados se parecen entre sí? | Tipos (≥ 3 sentencias) cuyo centroide queda más cerca de su familia que de otra | 166 de 246 |
| **Latencia p50 / p95** (`latencia`) | ¿Cuánto espera el usuario? | Mediana y percentil 95 por consulta | 30.8 s / 52.6 s, 5 llamadas al LLM (p50); con QC_1 apagado, como queda la app, 17.1 s / 34.6 s y 2 llamadas (`20261124-020004_2e50981_latencia`) |

In [19]:
# Evaluación final: partición de prueba, una sola corrida por comando, con el código y los datos congelados.
# Latencia, coherencia de las familias y robustez no dependen de la partición (latencia y robustez: la versión final).
from copiloto_juridico.evaluacion.reporte import ultimo_resultado

con_particion = {"recuperacion", "fichas-referencia", "esfuerzo", "qc"}
for comando in ["recuperacion", "fichas-referencia", "esfuerzo", "qc", "familias-coherencia", "latencia", "robustez"]:
    d = ultimo_resultado(comando, particion="prueba" if comando in con_particion else None)
    if d is None:
        continue
    display(Markdown(f"#### {comando} · {d['_archivo']}"))
    for nombre, t in d["tablas"].items():
        if not t.get("diagnostico"):
            display(Markdown(f"**{nombre}.** {t['descripcion']}"))
            display(pd.DataFrame(t["filas"], columns=t["columnas"]).round(3))

#### recuperacion · 20261121-032545_4b1529e_recuperacion.json

**Clasificación de la consulta.** Hit@3 = la categoría correcta queda dentro del filtro de 3 categorías (si no, ninguna etapa posterior la recupera). La sentencia de la que se redactó la consulta no participa (leave-one-out). IC95 de Wilson; F1 macro sobre la predicción top-1. k-NN = tipo mayoritario de las 10 sentencias más cercanas.

,variante,n,Hit@3 tipo,Hit@3 tipo IC95,Hit@3 familia,Hit@3 familia IC95,Top-1 tipo,Top-1 tipo IC95,Top-1 familia,Top-1 familia IC95,F1 macro tipo,F1 macro familia
0,Híbrido (app),46,0.717,"[0.575, 0.827]",0.717,"[0.575, 0.827]",NaN,—,0.370,"[0.245, 0.514]",NaN,0.290
1,k-NN 10 (línea base),46,0.087,"[0.034, 0.203]",0.435,"[0.302, 0.578]",0.065,"[0.022, 0.175]",0.326,"[0.209, 0.470]",0.029,0.319


**Errores de clasificación.** Consultas cuya familia correcta quedó fuera del top-3.

,consulta,partición,tipo correcto,familia,top-3
0,A008,prueba,penal/amenzas-y-lesiones,amenazas-y-extorsion,penal/familia/violencia-familiar · penal/familia/contra-la-dignidad-de-las-personas · penal/familia/lesiones
1,A018,prueba,penal/contra-la-dignidad-de-las-personas,contra-la-dignidad-de-las-personas,penal/familia/lesiones · penal/familia/violencia-familiar · penal/familia/homicidio
2,A027,prueba,penal/dano-en-las-cosas-y-otros,dano-en-las-cosas,penal/familia/delitos-culposos · penal/familia/contra-la-dignidad-de-las-personas · penal/familia/violenci...
3,A044,prueba,penal/desaparicion-forzada-de-personas-agravada,desaparicion-de-personas,penal/familia/privacion-de-la-libertad-y-secuestro · penal/familia/trata-de-personas · penal/familia/cuest...
4,A071,prueba,penal/lesiones-calificadas,lesiones,penal/familia/fraude-y-abuso-de-confianza · penal/familia/delitos-culposos · penal/familia/amenazas-y-exto...
5,A119,prueba,penal/violencia-familiar-y-violacion,violencia-familiar,penal/familia/delitos-sexuales · penal/familia/trata-de-personas · penal/familia/contra-la-dignidad-de-las...
6,A129,prueba,penal/amenazas,amenazas-y-extorsion,penal/familia/violencia-familiar · penal/familia/contra-la-dignidad-de-las-personas · penal/familia/delito...
7,A130,prueba,penal/amenazas,amenazas-y-extorsion,penal/familia/homicidio · penal/familia/contra-la-dignidad-de-las-personas · penal/familia/delitos-sexuales
8,A139,prueba,penal/contra-la-dignidad-de-las-personas,contra-la-dignidad-de-las-personas,penal/familia/robo · penal/familia/lesiones · penal/familia/feminicidio
9,A165,prueba,penal/despojo,despojo,penal/familia/fraude-y-abuso-de-confianza · penal/familia/robo · penal/familia/falsedad-y-falsificacion


**Matriz de confusión por familia (top-1).** Filas = familia correcta, columnas = familia predicha. La diagonal son aciertos.

... (33 outputs left out)

,etapa,n,p50,p95
0,Extracción de conceptos clave,10,0.818,1.174
1,Clasificación híbrida de la consulta,10,0.412,2.768
2,Búsqueda adaptativa de precedentes,10,0.000,0.000
3,"Análisis de precedentes (verificación, extracción e informe)",10,15.163,32.476
4,Síntesis comparativa final,10,16.019,19.419
5,Total por consulta,10,30.830,52.556


**Llamadas al LLM y descartes.** Por consulta ejecutada.

,consultas,sin resultado,llamadas p50,llamadas p95,llamadas promedio,descartes QC_2 delito de otra familia,veredictos ambiguos por consulta
0,10,0,5.0,6.0,5.2,0.1,0.0


#### robustez · 20261124-004225_82c94ed_robustez.json

### 9.2 Mejoras probadas después de la evaluación final

En la partición de desarrollo, por pares (IC 95 % por
bootstrap), sin volver a medir la de prueba. Se adoptó solo lo que mejoró sin empeorar otra medida:

| Cambio | Resultado | Decisión | Corrida |
|---|---|---|---|
| Síntesis con un ejemplo de líneas citadas y revisión final de las citas (19 consultas) | con un ejemplo de contenido jurídico, cita válida −0.024 [−0.144, +0.078], lo copia en 9 de 19 síntesis y pronostica en una (una primera medición, con otros precedentes, daba +0.086); con un ejemplo solo de formato, +0.024 [−0.069, +0.121], sin copias | descartado: la app sigue con la síntesis anterior | `20261124-003427_905c255_sintesis-citas` |
| QC_1 lee la ficha precalculada en lugar del fragmento (660 pares) | recall 0.725 → **0.844** (+0.120 [+0.036, +0.204]); especificidad 0.410 → 0.396 (no significativo); precisión 0.294 → 0.321 | adoptado | `20261123-230824_5fc9b67_qc` |
| QC_1 con `qwen2.5:7b` (660 pares) | recall 0.844 → 0.425 (−0.419, significativo); 1.9 s por llamada, pero cada cambio de modelo en 8 GB cuesta 5-8 s | descartado | `20261123-234717_e28a600_qc` |
| QC_1 solo con el veredicto, sin razonar (660 pares) | 0.59 s por llamada, pero responde SI en 649 de 660 pares (especificidad 0.022): deja de filtrar | descartado | `20261123-234717_e28a600_qc` |
| Ficha: pedir solo los artículos que invoca la Sala (120 sentencias) | precisión del fundamento +0.010 [−0.040, +0.060]; exhaustividad −0.023, significativa | descartado | `20261123-174502_b8fdd94_fundamento-prompt` |
| Ficha: darle al modelo la parte considerativa (120 sentencias) | precisión −0.048 [−0.096, +0.006]; exhaustividad −0.030, significativa | descartado | `20261123-212426_5fc9b67_fundamento-contexto` |
| Consultas planteadas como toca en estudio (39 consultas) | Hit@3 de familia igual (+0.051 [−0.077, +0.205]); nDCG@5 0.266 → 0.145, significativo | medición del perfil | `20261123-202205_e566e3d-sucio_perfil-tribunal` |


In [20]:
corridas = []
for p in sorted(Path("evaluacion/resultados").glob("*.json")):
    d = json.loads(p.read_text())
    corridas.append({"comando": d["comando"], "corrida": p.stem, "fecha": d["fecha"][:16], "commit": d["commit"],
                     "partición de prueba": "sí" if (d.get("parametros") or {}).get("particion") == "prueba" else ""})
pd.DataFrame(corridas).groupby("comando").tail(1).set_index("comando")  # la más reciente de cada experimento

,corrida,fecha,commit,partición de prueba
comando,,,,
qc-diagnostico,20261113-114412_fb1f3fe_qc-diagnostico,2026-11-13T11:44,fb1f3fe,
modelos,20261113-125326_fb1f3fe_modelos,2026-11-13T12:53,fb1f3fe,
fichas-diagnostico,20261121-020509_d32154f_fichas-diagnostico,2026-11-21T02:05,d32154f,
metadatos-cobertura,20261121-022448_b2754ef_metadatos-cobertura,2026-11-21T02:24,b2754ef,
referencias,20261121-031724_723fe6b_referencias,2026-11-21T03:17,723fe6b,
recuperacion,20261121-032545_4b1529e_recuperacion,2026-11-21T03:25,4b1529e,sí
fichas-referencia,20261121-032553_4b1529e_fichas-referencia,2026-11-21T03:25,4b1529e,sí
esfuerzo,20261121-032625_4b1529e_esfuerzo,2026-11-21T03:26,4b1529e,sí
familias-coherencia,20261121-032630_4b1529e_familias-coherencia,2026-11-21T03:26,4b1529e,


## 10. Confidencialidad, alcance y líneas de desarrollo

**Confidencialidad.** La aplicación se ejecuta en el equipo del usuario; la consulta no se transmite a terceros. Los
datos personales que el STJ suprimió en la versión pública ([eliminado]) nunca se infieren. Los resultados guardados
para la demostración pueden desactivarse (`COPILOTO_CACHE_RESULTADOS=false`).

**Alcance y líneas de desarrollo.**

- Los dos perfiles de usuario comparten la búsqueda y las fichas evaluadas; las consultas de evaluación están
  redactadas como las plantearía un abogado, por lo que la medición con el planteamiento de un toca en estudio
  (tribunal de alzada) es una línea de desarrollo.
- Para el tribunal de alzada, la integración de la búsqueda por hechos y de la ficha en el portal del STJ resolvería
  el problema de localización en su origen.

- La referencia de la evaluación proviene del tribunal y del texto de la resolución, no del juicio de un experto sobre
  la utilidad de cada precedente; acto impugnado, apelante y sentido se contrastan con las reglas sobre el texto.
- Las familias de delito se asignan por reglas; su revisión por un especialista es una línea de desarrollo.
- El acuerdo entre lectores sobre la relevancia graduada es leve (kappa cercano a 0.2; corrida
  `20261117-065346_a60ca4f_referencias`); por ello las métricas principales se apoyan en la etiqueta del STJ.
- 54 resoluciones penales conservan la versión con páginas omitidas; su sustitución es una línea de desarrollo.
- Varios tipos penales cuentan con muy pocas resoluciones; cuando ningún precedente corresponde a la familia
  detectada, la aplicación lo advierte.
- El aviso de consulta ajena a la materia penal se basa en una lista de términos.
- Los documentos escaneados se leen por OCR (Tesseract) al construir el corpus; solo 1 de los 1,899 PDF lo está y no
  se incorporó, porque su versión escaneada no tiene suprimidos los datos personales.

**Decisión de alcance: solo materia penal.** El corpus inicial incluía 2,119 resoluciones de otras materias
(civil, familiar, mercantil y laboral). Todo el método vigente es específico de la materia penal: familias de delito,
ficha de 14 campos (acto impugnado, agravios, sentido del fallo), reglas de lectura de los puntos resolutivos,
consultas de prueba y evaluación. Por ello esas resoluciones se retiraron del corpus y del índice (186,971 fragmentos
penales de 211,572), y con ellas las instrucciones de otras materias.

## 11. Referencias

1. Tom's Hardware, «Save $300 on Asus' 2024 ROG Zephyrus G14 with OLED display and RTX 4060» (precio de lista y de
   oferta). https://www.tomshardware.com/laptops/gaming-laptops/save-dollar300-on-asus-2024-rog-zephyrus-g14-with-oled-display-and-rtx-4060
2. ASUS, especificaciones del ROG Zephyrus G14 (2024) GA403. https://rog.asus.com/us/laptops/rog-zephyrus/rog-zephyrus-g14-2024/spec
3. Notebookcheck, «NVIDIA GeForce RTX 4060 Laptop GPU – Benchmarks and Specs».
   https://www.notebookcheck.net/NVIDIA-GeForce-RTX-4060-Laptop-GPU-Benchmarks-and-Specs.675692.0.html
4. Wang, L. et al. (2024). *Multilingual E5 Text Embeddings: A Technical Report*. arXiv:2402.05672.
   https://arxiv.org/abs/2402.05672 · Modelo: https://huggingface.co/intfloat/multilingual-e5-base
5. Meta (2024). Llama 3.1 8B Instruct. https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct · Ollama:
   https://ollama.com/library/llama3.1 · https://ollama.com/library/gemma2 · https://ollama.com/library/mistral
6. Malkov, Y. y Yashunin, D. (2018). *Efficient and robust approximate nearest neighbor search using Hierarchical
   Navigable Small World graphs*. arXiv:1603.09320. https://arxiv.org/abs/1603.09320 · Chroma: https://docs.trychroma.com/
7. Robertson, S. y Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*. Foundations and
   Trends in Information Retrieval 3(4). https://doi.org/10.1561/1500000019
8. Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*. arXiv:2306.05685.
   https://arxiv.org/abs/2306.05685
9. Panickssery, A., Bowman, S. y Feng, S. (2024). *LLM Evaluators Recognize and Favor Their Own Generations*.
   arXiv:2404.13076. https://arxiv.org/abs/2404.13076
10. Cormack, G., Clarke, C. y Büttcher, S. (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual Rank
    Learning Methods*. SIGIR '09. https://doi.org/10.1145/1571941.1572114
11. Cohen, J. (1960). *A Coefficient of Agreement for Nominal Scales*. Educational and Psychological Measurement
    20(1). https://doi.org/10.1177/001316446002000104
12. México Evalúa (2026). *Radiografía de la Impunidad 2024: Hallazgos del Sistema de Justicia Penal en México*.
    https://mexicoevalua.org/solo-1-de-cada-10-casos-se-resuelve-en-mexico-radiografia-de-la-impunidad-propone-soluciones-para-fortalecer-la-justicia-penal/
13. World Justice Project (2025). *Rule of Law Index 2025*, perfil de México.
    https://worldjusticeproject.org/rule-of-law-index/s/pdfs/2025-Mexico.pdf
14. IIEG Jalisco (2025). *Censo Nacional de Impartición de Justicia Estatal 2025: principales resultados de Jalisco*
    (datos de INEGI, 2024). https://iieg.gob.mx/ns/wp-content/uploads/2025/11/ficha_censo_nacional_imparticion_justicia_estatal_jalisco_2025.pdf

---
*Herramienta académica de apoyo basada en resoluciones públicas del Supremo Tribunal de Justicia del Estado de
Jalisco. No constituye asesoría jurídica ni sustituye la lectura de la resolución original ni el criterio profesional
del abogado.*